In [ ]:
pip install -U transformers

In [1]:
# IMPORTS

from __future__ import annotations
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

C:\Users\Fcomm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class LLM:
    def __init__(self):
        model_name = "google/flan-t5-small"
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        print("Loading model...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def ask(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt")
        outputs = self.model.generate(**inputs, max_length=50)
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response

# create instance
llm = LLM()

Loading tokenizer...


Loading model...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 6795.46it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
prompt = """
         You are an agent in a grid world with a primary mission to find and step on the goal tile, and a secondary mission to find the most optimal sequence of moves to achieve this goal. You have a limited field of vision, perceiving the tiles directly ahead, two tiles ahead, to your left, to your right, and diagonally to your left and right. You cannot move through wall tiles, but you can move on floor tiles.

**Environment State:**

*   **Ahead:** Wall
*   **Two Tiles Ahead:** Wall
*   **Left:** Floor
*   **Right:** Wall
*   **Diagonal Left:** Wall
*   **Diagonal Right:** Floor

**Objective:** Determine and output the single most optimal action to advance towards the goal, considering the current environment state and the need for optimal pathfinding.

**Available Actions:**
*   `forward`
*   `left`
*   `right`

# Reasoning Steps

1.  **Analyze immediate surroundings:** Evaluate the accessibility of tiles directly in front, to the left, and to the right.
2.  **Consider future obstacles:** Assess how immediate moves might lead into or away from further obstacles (walls, etc.).
3.  **Evaluate diagonal options:** Determine if diagonal moves offer a better path or reveal more accessible tiles.
4.  **Prioritize goal-oriented movement:** Consider which action is most likely to lead towards the goal, assuming the goal is not immediately blocked by walls in all directions.
5.  **Identify immediate path obstructions:** Note that moving `forward` is blocked by a wall directly ahead and two tiles ahead. Moving `left` is blocked by a wall to the immediate left.
6.  **Identify available paths:** The `right` direction is blocked by a wall. The `diagonal right` shows a floor tile, implying a possible path if a turn is made.
7.  **Determine optimal move:** Given the walls directly ahead and to the left, and the wall to the right, the only immediately viable direction that doesn't lead into a wall is to turn towards the `diagonal right` floor tile. This requires a `right` turn.

# Notes

*   The "most optimal" move implies a greedy approach that considers immediate best progress towards an unknown goal, while avoiding immediate impassable terrain.
        """
response = llm.ask(prompt)
print(response)

Token indices sequence length is longer than the specified maximum sequence length for this model (532 > 512). Running this sequence through the model will result in indexing errors


**Identify the most optimal move to advance towards the goal. **Identify the most optimal move to advance towards the goal. **Identify the most optimal move to advance towards the goal. **Identify the most optimal move to advance towards the goal. **


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
from datetime import datetime


class LLMRunner:
    def __init__(self, model_name):
        self.model_name = model_name

        print(f"[INFO] Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            low_cpu_mem_usage=True
        )

        self.model.eval()

    def run(self, prompt, max_new_tokens=50, temperature=0.7, top_p=0.9):
        inputs = self.tokenizer(prompt, return_tensors="pt")
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p
            )

        output_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        result = {
            "model": self.model_name,
            "timestamp": datetime.now().isoformat(),
            "input_prompt": prompt,
            "input_tokens": int(input_len),
            "output_text": output_text,
            "output_tokens": int(outputs.shape[1])
        }

        return result

    def save(self, result, filename="llm_results.jsonl"):
        with open(filename, "a", encoding="utf-8") as f:
            f.write(json.dumps(result) + "\n")

In [ ]:
#DO NOT RUN
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

#Failed to Load
"EleutherAI/pythia-410m"
"microsoft/phi-2"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "EleutherAI/gpt-neo-125M"    # swap models here

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True
)

# your existing prompt
text = prompt  

# tokenize (NO truncation since you need full input)
inputs = tokenizer(text, return_tensors="pt")

print("Token length:", inputs["input_ids"].shape[1])

print("Generating...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,   # safer than max_length
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

print("\n--- OUTPUT ---\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading tokenizer...


Loading model...


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 2195.87it/s]
GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Token length: 560
Generating...

--- OUTPUT ---


         You are an agent in a grid world with a primary mission to find and step on the goal tile, and a secondary mission to find the most optimal sequence of moves to achieve this goal. You have a limited field of vision, perceiving the tiles directly ahead, two tiles ahead, to your left, to your right, and diagonally to your left and right. You cannot move through wall tiles, but you can move on floor tiles.

**Environment State:**

*   **Ahead:** Wall
*   **Two Tiles Ahead:** Wall
*   **Left:** Floor
*   **Right:** Wall
*   **Diagonal Left:** Wall
*   **Diagonal Right:** Floor

**Objective:** Determine and output the single most optimal action to advance towards the goal, considering the current environment state and the need for optimal pathfinding.

**Available Actions:**
*   `forward`
*   `left`
*   `right`

# Reasoning Steps

1.  **Analyze immediate surroundings:** Evaluate the accessibility of tiles directly in front, to the l

In [ ]:
runner = LLMRunner("EleutherAI/gpt-neo-1.3B")

result = runner.run(prompt)

print("LLM Answer")
print(result["output_text"])

[INFO] Loading model: EleutherAI/gpt-neo-1.3B


Loading weights: 100%|██████████| 316/316 [00:00<00:00, 4665.72it/s]
GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-1.3B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...23}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...22}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



         You are an agent in a grid world with a primary mission to find and step on the goal tile, and a secondary mission to find the most optimal sequence of moves to achieve this goal. You have a limited field of vision, perceiving the tiles directly ahead, two tiles ahead, to your left, to your right, and diagonally to your left and right. You cannot move through wall tiles, but you can move on floor tiles.

**Environment State:**

*   **Ahead:** Wall
*   **Two Tiles Ahead:** Wall
*   **Left:** Floor
*   **Right:** Wall
*   **Diagonal Left:** Wall
*   **Diagonal Right:** Floor

**Objective:** Determine and output the single most optimal action to advance towards the goal, considering the current environment state and the need for optimal pathfinding.

**Available Actions:**
*   `forward`
*   `left`
*   `right`

# Reasoning Steps

1.  **Analyze immediate surroundings:** Evaluate the accessibility of tiles directly in front, to the left, and to the right.
2.  **Consider future obst

In [ ]:
#DO NOT RUN
runner = LLMRunner("microsoft/phi-2")

result = runner.run(prompt)

print("LLM Answer")
print(result["output_text"])

[INFO] Loading model: microsoft/phi-2


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b")

input_text = prompt
input_ids = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**input_ids)
print(tokenizer.decode(outputs[0]))

In [ ]:
#DO NOT RUN

from transformers import pipeline
import torch

model_id = "openai/gpt-oss-120b"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto",
)

messages = [
    {"role": "user", "content": "Explain quantum mechanics clearly and concisely."},
]

outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

In [ ]:
from clarifai.client import Model

model = Model(url="https://clarifai.com/moonshotai/kimi/models/Kimi-K2-Thinking")

chat_history = [
    {"role": "user", "content": "I am preparing for a systems design interview."},
    {"role": "assistant", "content": "Great. What topic do you want to practice first?"},
]

result = model.predict(prompt, chat_history=chat_history, max_tokens=2048)
print(result)


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_API_KEY",
    base_url="https://api.moonshot.ai/v1"  # Kimi endpoint
)

response = client.chat.completions.create(
    model="kimi-k2-thinking",
    messages= prompt
)

print(response.choices[0].message.content)

In [ ]:
pip install openai